# Coordinate arrays
Declare grids without field data and recover coordinates on demand.

In [ ]:
import numpy as np
import earth2studio as e2s

signature = e2s.coord_array(
    dims=("batch", "time", "lead_time", "variable", "lat", "lon"),
    coords={"lead_time": [np.timedelta64(0, "h")], "variable": ["u10m", "t2m"]},
    dynamic=("batch", "time"), grid="latlon-0.25deg",
    statistics={"u10m": "mean:24h"},
)
signature

In [ ]:
signature.dims, signature.shape, signature.data.nbytes, signature.e2s.get_grid()

## Recover coordinates

In [ ]:
populated = signature.e2s.materialize_grid_coords()
populated.coords

## Explicit coordinates

In [ ]:
explicit = e2s.coord_array(
    dims=("variable", "lat", "lon"),
    coords={"variable": ["t2m"], "lat": np.linspace(90, -90, 721), "lon": np.arange(0, 360, 0.25)},
    grid="latlon-0.25deg",
)
explicit.coords

## HRRR and HPX

In [ ]:
hrrr = e2s.coord_array(
    dims=("batch", "variable", "hrrr_y", "hrrr_x"),
    coords={"variable": ["u10m"]}, dynamic=("batch",), grid="hrrr",
).e2s.materialize_grid_coords()
hpx = e2s.coord_array(
    dims=("batch", "variable", "hpx"),
    coords={"variable": ["u10m"]}, dynamic=("batch",), grid="hpx6",
).e2s.materialize_grid_coords()
[(array.e2s.get_grid(), tuple(array.coords), array.data.nbytes) for array in (hrrr, hpx)]

## One-step output

In [ ]:
output = signature.assign_coords(lead_time=signature.lead_time + np.timedelta64(6, "h"))
output.dims, output.lead_time.values, output.data.nbytes